# Lowa Liquor Sales Data Analysis (2024-2025)


This notebook presents an analytical project based on the Iowa Liquor Sales dataset, focusing exclusively on the years 2024 and 2025. The goal is to demonstrate how public data can be integrated into Google BigQuery, processed efficiently, and connected to Jupyter for exploratory analysis and visualization.

## Project Scope
- Extract and store relevant data (2024–2025) from the public Iowa Liquor Sales dataset into a personal BigQuery project.
- Perform queries to identify sales trends by county, store, and product category.
- Summarize monthly and yearly sales to highlight consumption patterns.
- Visualize results using Python libraries such as Pandas, Numpy, Matplotlib, and Seaborn.

## Purpose
The project aims to showcase practical skills in:
- Cloud data management with BigQuery.
- SQL querying and dataset optimization.
- Python-based data analysis and visualization.
- Building a clear, reproducible workflow for presentation in a professional portfolio.

By the end of this notebook, readers will understand the main sales trends in Iowa during 2024–2025 and see how cloud-based tools can be combined with Jupyter to deliver meaningful insights.


## Import libraries
In this section, we import the essential Python libraries for data analysis and visualization: Pandas, NumPy, Matplotlib, and Seaborn.


In [29]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os
from google.cloud import bigquery



## Connect to BigQuery 
In this section, we establish the connection to Google BigQuery in order to load the Iowa Liquor Sales dataset (2024–2025) into our notebook for analysis.

In [30]:
import os
from google.cloud import bigquery
import pandas as pd

# Set the environment variable for your service account JSON
# Use forward slashes (/) to avoid escape sequence issues in Windows paths
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\anton\OneDrive\Documentos\keydatasetsale\dalesdataanalysis-d7e3cbd72d11.json"

# Create a BigQuery client
client = bigquery.Client()

# Define your SQL query
query = """
SELECT *
FROM `dalesdataanalysis.sales_data.iowa_sales_2024_2025`
WHERE RAND() < 0.02
"""

# Run the query and load results into a Pandas DataFrame
df = client.query(query).to_dataframe()

# Preview the dataset
df.head(10)


C:\Users\anton\OneDrive\Documentos\proyectodata\salesdateanalysis\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date,store_name,city,county,category_name,item_description,bottles_sold,sale_dollars,volume_sold_liters
0,2024-11-13,FAREWAY STORES #827 / CENTERVILLE,CENTERVILLE,APPANOOSE,IMPORTED VODKAS,BELVEDERE PURE,-6,-148.44,-4.50
1,2025-06-20,BARRYS MINI MART,HARPERS FERRY,ALLAMAKEE,TENNESSEE WHISKIES,OLE SMOKY TENNESSEE SALTY WATERMELON WHISKEY MINI,1,26.25,0.05
2,2025-06-13,BARRYS MINI MART,HARPERS FERRY,ALLAMAKEE,TEMPORARY & SPECIALTY PACKAGES,WHISTLEPIG RYE PIGLETS 6/10/12 MINI,1,18.75,0.05
3,2024-10-30,BIG 10 MART #19 / HIAWATHA,HIAWATHA,LINN,CANADIAN WHISKIES,BLACK VELVET MINI,1,13.31,0.05
4,2025-06-02,CALAMUS COUNTRY STORE / CALAMUS,CALAMUS,CLINTON,IRISH WHISKIES,JAMESON MINI,1,19.80,0.05
5,2025-01-02,CASEY'S #3920 / MOUNT PLEASANT,MOUNT PLEASANT,HENRY,AMERICAN SCHNAPPS,99 STRAWBERRIES MINI,1,78.60,0.05
6,2024-12-09,CASEY'S #4396 / BONDURANT,BONDURANT,POLK,AMERICAN CORDIALS & LIQUEURS,99 WATERMELON PET MINI,1,78.60,0.05
7,2024-02-15,CASEY'S GENERAL STORE # 1896/ CLEAR LAKE,CLEARLAKE,CERRO GORDO,AMERICAN CORDIALS & LIQUEURS,99 WATERMELON PET MINI,1,77.40,0.05
8,2025-07-09,CASEY'S GENERAL STORE # 2687/ HAWARDEN,HAWARDEN,SIOUX,AMERICAN CORDIALS & LIQUEURS,99 PARTY BUCKET MINIS,1,78.60,0.05
9,2025-07-18,CASEY'S GENERAL STORE #1126 / WYOMING,WYOMING,JONES,100% AGAVE TEQUILA,MARGARITAVILLE GOLD TEQUILA MINI,1,9.72,0.05


## General data frame infomracion 
Initial overview of the DataFrame structure and characteristics, including column types, record counts, missing values, and basic data .

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102023 entries, 0 to 102022
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   date                102023 non-null  dbdate 
 1   store_name          102023 non-null  str    
 2   city                101985 non-null  str    
 3   county              101985 non-null  str    
 4   category_name       102023 non-null  str    
 5   item_description    102023 non-null  str    
 6   bottles_sold        102023 non-null  Int64  
 7   sale_dollars        102023 non-null  float64
 8   volume_sold_liters  102023 non-null  float64
dtypes: Int64(1), dbdate(1), float64(2), str(5)
memory usage: 15.3 MB


In this stage, the date column was converted into the datetime type using the pd.to_datetime() function with the parameter errors='coerce'. This ensures that valid values are transformed into usable dates for temporal analysis, while invalid formats are safely converted into NaT without interrupting the workflow. A basic data cleaning process was then applied: removal of duplicate records, handling of missing values in the city and county columns, normalization of text in store names and categories, and verification of outliers in sales and volume metrics. These actions guarantee that the dataset is structured, consistent, and ready for exploratory analysis and visualization.

In [41]:
# 1.- Convert data column to datatime 
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 2.- Handle missing values in 'city' and 'country'
#Options A: Drop rows with nulls 

df = df.dropna(subset=['city','county'])

# 3.- Remove duplicate rows 
df = df.drop_duplicates()


# 4.- Normalize text field (strps, space, lowercase)
df['store_name'] = df['store_name'].str.strip().str.lower()
df['category_name'] = df['category_name'].str.strip().str.lower()
df['city'] = df['city'].str.strip().str.lower()
df['county'] = df['county'].str.strip().str.lower()
df['item_description'] = df['item_description'].str.strip().str.lower()

# Remove otliers or invalid values in numeric columns 
df = df[df['sale_dollars'] >= 0]
df = df[df['volume_sold_liters'] >= 0]



## Cleaned Dataset Overview

Final rows: 101,867 records (after removing nulls and duplicates).

Columns: 9 total, with corrected data types.

Date column: successfully converted to datetime64[ns], ready for time-based analysis (year, month, day).

Text fields: (store_name, city, county, category_name, item_description) normalized to lowercase and stripped of extra spaces.

Numeric fields: (bottles_sold, sale_dollars, volume_sold_liters) validated, with negative or invalid values removed.

Consistency:

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 101867 entries, 1 to 102022
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype        
---  ------              --------------   -----        
 0   date                101867 non-null  datetime64[s]
 1   store_name          101867 non-null  str          
 2   city                101867 non-null  str          
 3   county              101867 non-null  str          
 4   category_name       101867 non-null  str          
 5   item_description    101867 non-null  str          
 6   bottles_sold        101867 non-null  Int64        
 7   sale_dollars        101867 non-null  float64      
 8   volume_sold_liters  101867 non-null  float64      
dtypes: Int64(1), datetime64[s](1), float64(2), str(5)
memory usage: 16.1 MB


In [43]:
df.head(10)

,date,store_name,city,county,category_name,item_description,bottles_sold,sale_dollars,volume_sold_liters
1,2025-06-20,barrys mini mart,harpers ferry,allamakee,tennessee whiskies,ole smoky tennessee salty watermelon whiskey mini,1,26.25,0.05
2,2025-06-13,barrys mini mart,harpers ferry,allamakee,temporary & specialty packages,whistlepig rye piglets 6/10/12 mini,1,18.75,0.05
3,2024-10-30,big 10 mart #19 / hiawatha,hiawatha,linn,canadian whiskies,black velvet mini,1,13.31,0.05
4,2025-06-02,calamus country store / calamus,calamus,clinton,irish whiskies,jameson mini,1,19.80,0.05
5,2025-01-02,casey's #3920 / mount pleasant,mount pleasant,henry,american schnapps,99 strawberries mini,1,78.60,0.05
6,2024-12-09,casey's #4396 / bondurant,bondurant,polk,american cordials & liqueurs,99 watermelon pet mini,1,78.60,0.05
7,2024-02-15,casey's general store # 1896/ clear lake,clearlake,cerro gordo,american cordials & liqueurs,99 watermelon pet mini,1,77.40,0.05
8,2025-07-09,casey's general store # 2687/ hawarden,hawarden,sioux,american cordials & liqueurs,99 party bucket minis,1,78.60,0.05
9,2025-07-18,casey's general store #1126 / wyoming,wyoming,jones,100% agave tequila,margaritaville gold tequila mini,1,9.72,0.05
10,2024-09-27,casey's general store #1503 / tabor,tabor,fremont,american schnapps,99 root beer mini,1,77.40,0.05


In [44]:
df.describe()

,date,bottles_sold,sale_dollars,volume_sold_liters
count,101867,101867.0,101867.000000,101867.000000
mean,2024-12-30 00:41:46,12.22388,172.861372,9.003949
min,2024-01-02 00:00:00,1.0,1.500000,0.020000
25%,2024-06-29 00:00:00,3.0,51.750000,1.500000
50%,2024-12-23 00:00:00,6.0,90.360000,4.500000
75%,2025-06-30 00:00:00,12.0,171.000000,9.600000
max,2025-12-31 00:00:00,1800.0,56871.360000,3150.000000
std,NaN,30.135228,579.118670,37.377662
